<a href="https://colab.research.google.com/github/ritvik-123/EMP-Project/blob/master/Module_1_26.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
import os
import re
import json
import time
import joblib
import pandas as pd
import numpy as np

from tqdm.auto import tqdm

import torch

from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, multilabel_confusion_matrix

import nltk
nltk.download("punkt")
nltk.download("punkt_tab")
from nltk.tokenize import sent_tokenize

from google import genai
from google.genai import types
import os
import getpass


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
CSV_PATH = "../Data/"

MODEL_PATH = "../Model/"


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:

%pwd
%cd /content/drive/MyDrive/EMP/Notebook

/content/drive/MyDrive/EMP/Notebook


In [5]:

%pwd

'/content/drive/MyDrive/EMP/Notebook'

In [28]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

if device == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

Device: cuda
GPU: Tesla T4


In [6]:
df = pd.read_csv(CSV_PATH + "Module 1.csv")
print(df.shape)
df.head()

(27, 3)


,ID,Campus,Comment
0,32,Bakersfield,NaN
1,36,Chico,Something that I noticed is how I feel about m...
2,128,Chico,"As an intersectional scholar, I had experience..."
3,25,Dominguez Hills,It was great being able to reflect on the iden...
4,91,Dominguez Hills,I noticed that it was easy for me to identify ...


In [7]:
df = df.drop(columns = ['ID'])

In [8]:
df.columns.tolist()

['Campus', 'Comment']

In [9]:
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = text.replace("\r", " ").replace("\n"," ")
    text = re.sub(r"\s+", " ", text).strip()
    return text

def split_into_sentences(text):
    text = clean_text(text)
    if not text:
        return []

    sentences = sent_tokenize(text)
    sentences = [s.strip() for s in sentences if s.strip()]
    return sentences

sentence_rows = []

for source_id, row in df.reset_index(drop=True).iterrows():
    campus = row.get("Campus", "")
    comment = row.get("Comment", "")
    sentences = split_into_sentences(comment)

    for sentence_index, sentence in enumerate(sentences):
        sentence_rows.append({
            "source_id": source_id,
            "campus": campus,
            "sentence_index": sentence_index,
            "sentence_id": f"{source_id}_{sentence_index}",
            "sentence": sentence
        })
sent_df = pd.DataFrame(sentence_rows)

print(sent_df.shape)
sent_df.head(10)

(154, 5)


,source_id,campus,sentence_index,sentence_id,sentence
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...
1,1,Chico,1,1_1,I think it is important to know how each indiv...
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...
3,1,Chico,3,1_3,Since we all have so many different identities...
4,1,Chico,4,1_4,I want to continue listening to my students in...
5,2,Chico,0,2_0,"As an intersectional scholar, I had experience..."
6,2,Chico,1,2_1,"However, one highlight of this exercise for me..."
7,2,Chico,2,2_2,To be able to list the various identities with...
8,2,Chico,3,2_3,An insight I gained from this activity has to ...
9,2,Chico,4,2_4,I identify as agnostic and never really consid...


In [10]:
sent_df.to_csv(CSV_PATH + "Module 1 sentence.csv", index = False)
print(f"saved sentence-level data to: {CSV_PATH}")

saved sentence-level data to: ../Data/


In [11]:
import getpass
import os
from google import genai

os.environ["GEMINI_API_KEY"] = getpass.getpass("Enter Gemini API key: ")

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

for model in client.models.list():
    print(model.name)
    print("supported actions:", getattr(model, "supported_actions", None))
    print("-" * 80)

Enter Gemini API key: ··········
models/gemini-2.5-flash
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.5-pro
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash-001
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
--------------------------------------------------------------------------------
models/gemini-2.0-flash-lite-001
supported actions: ['generateContent', 'countTokens', 'createCachedContent', 'batchGenerateContent']
-------

In [12]:
GEMINI_MODEL = "gemini-2.5-flash-lite"

In [13]:
LABELS = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]

LABEL_DEFINITIONS = """
You are labeling sentences for a qualitative research project.

Classify how much the sentence leans toward each of four forms of oppression.

1. Ideological oppression:
Belief systems, dominant ideas, stereotypes, norms, assumptions, cultural narratives,
or values that justify inequality.

2. Institutionalized oppression:
Oppression embedded in systems, institutions, policies, schools, religion, workplaces,
government, laws, programs, access to resources, or formal structures.

3. Interpersonal oppression:
Oppression expressed through person-to-person interaction, exclusion, judgment,
bias, conflict, mistreatment, microaggressions, or treatment by others.

4. Internalized oppression:
Oppression absorbed into the self, including shame, self-doubt, hiding identity,
discomfort with identity, accepting negative beliefs about oneself, or seeing oneself
as lesser due to dominant narratives.

For each oppression type, return:
- a binary value: 1 if the sentence clearly leans toward that oppression type, otherwise 0
- a score from 0.0 to 1.0 showing how strongly the sentence leans toward that type

Important:
- Do not force a label.
- If the sentence does not clearly lean toward any of the four oppression types, all binary labels should be 0.
- Activity feedback alone should usually receive all 0s.
- General identity reflection without oppression should usually receive all 0s.
- Teaching application should only receive a label if it clearly mentions systems, bias, exclusion, inequity, or oppression.
- Do not create a fifth category.
"""

In [14]:
def build_prompt(sentence):
    return f"""
{LABEL_DEFINITIONS}

Return only valid JSON in this exact format:

{{
  "ideological": 0,
  "institutionalized": 0,
  "interpersonal": 0,
  "internalized": 0,
  "ideological_score": 0.0,
  "institutionalized_score": 0.0,
  "interpersonal_score": 0.0,
  "internalized_score": 0.0,
  "primary_leaning": "none",
  "rationale": "brief explanation"
}}

Sentence:
\"\"\"{sentence}\"\"\"
"""

def safe_json_parse(text):
    """
    Tries to parse Gemini output as JSON
    Also handles cases where the model wraps JSON in markdown.
    """

    if text is None:
        return None

    text = text.strip()

    # Remove markdown fences if present
    text = re.sub(f"^```json","",text).strip()
    text = re.sub(r"^```", "", text).strip()
    text = re.sub(r"```$","", text).strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass

    # Try extracting the first JSON object
    match = re.search(r"\{.*\}", text, flags=re.DOTALL)
    if match:
        try:
            return json.loads(match.group(0))
        except json.JSONDecodeError:
            return None

    return None

def label_sentence_with_gemini(sentence, max_retries = 3, sleep_seconds = 2):
    prompt = build_prompt(sentence)

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model = GEMINI_MODEL,
                contents = prompt,
                config=types.GenerateContentConfig(
                    temperature  = 0,
                    response_mime_type="application/json"
                )
            )

            parsed = safe_json_parse(response.text)

            if parsed is None:
                raise ValueError(f"Could not parse JSON: {response.text}")

            # normalize labels
            result = {}
            for label in LABELS:
                # Binary 0/1 label
                result[label] = int(parsed.get(label, 0))

                # Leaning score
                score_col = f"{label}_score"
                result[score_col] = float(parsed.get(score_col, 0.0))

            # If no label is selected, mark none or unclear
            if sum(result[label] for label in LABELS) == 0:
                result["primary_leaning"] = "none"

            else:
                result["primary_leaning"] = max(
                    LABELS,
                    key=lambda label: result[f"{label}_score"]
                )

            result["rationale"] = parsed.get("rationale", "")
            result["gemini_raw"] = response.text

            return result

        except Exception as e:
            if attempt == max_retries - 1:
                result = {}

                for label in LABELS:
                    result[label] = 0
                    result[f"{label}_score"] = 0.0

                result["primary_leaning"] = "none"
                result["rationale"]  = f"ERROR: {str(e)}"
                result["gemini_raw"] = ""

                return result;

            time.sleep(sleep_seconds * (attempt + 1))

In [15]:
sample_df = sent_df.copy()

In [16]:
labeled_rows = []

for _,row in tqdm(sample_df.iterrows(), total = len(sample_df)):
    sentence = row["sentence"]
    labels = label_sentence_with_gemini(sentence)

    combined = row.to_dict()
    combined.update(labels)

    # manual review fields
    combined["reviewed"] = 0
    combined["review_notes"] = ""

    labeled_rows.append(combined)

gemini_df = pd.DataFrame(labeled_rows)

gemini_df

  0%|          | 0/154 [00:00<?, ?it/s]

,source_id,campus,sentence_index,sentence_id,sentence,ideological,ideological_score,institutionalized,institutionalized_score,interpersonal,interpersonal_score,internalized,internalized_score,primary_leaning,rationale,gemini_raw,reviewed,review_notes
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...,0,0.0,0,0.0,0,0.0,1,0.7,internalized,The sentence reflects on a shift in self-perce...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
1,1,Chico,1,1_1,I think it is important to know how each indiv...,0,0.0,0,0.0,1,0.7,0,0.0,interpersonal,The sentence emphasizes the importance of indi...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence describes differing life experien...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
3,1,Chico,3,1_3,Since we all have so many different identities...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence discusses the complexity of inter...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
4,1,Chico,4,1_4,I want to continue listening to my students in...,0,0.0,1,0.7,0,0.0,0,0.0,institutionalized,The sentence discusses interpreting generalize...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
149,22,San Francisco,0,22_0,"Age: 56 years, my son calls me old a number of...",0,0.0,0,0.0,1,0.7,0,0.0,interpersonal,The sentence describes a person being called '...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
150,23,San Francisco,0,23_0,"Status: Lecturer Faculty, Instructor",0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence simply states a job title and doe...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
151,24,San Francisco,0,24_0,Gender: Male,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence is a simple statement of gender a...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,
152,25,San Francisco,0,25_0,Disability: Wearing eye glasses,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence simply states a fact about disabi...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,


In [17]:
gemini_df.to_csv(CSV_PATH + "Module 1 Sentences Gemini.csv", index = False)
print(f"Saved Gemini weka labels to: {CSV_PATH}")

Saved Gemini weka labels to: ../Data/


In [18]:
reviewed_df = pd.read_csv(CSV_PATH + "Module 1 Sentences Gemini.csv")

print(reviewed_df.shape)
reviewed_df.head()

(154, 18)


,source_id,campus,sentence_index,sentence_id,sentence,ideological,ideological_score,institutionalized,institutionalized_score,interpersonal,interpersonal_score,internalized,internalized_score,primary_leaning,rationale,gemini_raw,reviewed,review_notes
0,1,Chico,0,1_0,Something that I noticed is how I feel about m...,0,0.0,0,0.0,0,0.0,1,0.7,internalized,The sentence reflects on a shift in self-perce...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
1,1,Chico,1,1_1,I think it is important to know how each indiv...,0,0.0,0,0.0,1,0.7,0,0.0,interpersonal,The sentence emphasizes the importance of indi...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
2,1,Chico,2,1_2,Our students (and everyone around us) are havi...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence describes differing life experien...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
3,1,Chico,3,1_3,Since we all have so many different identities...,0,0.0,0,0.0,0,0.0,0,0.0,none,The sentence discusses the complexity of inter...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN
4,1,Chico,4,1_4,I want to continue listening to my students in...,0,0.0,1,0.7,0,0.0,0,0.0,institutionalized,The sentence discusses interpreting generalize...,"{\n ""ideological"": 0,\n ""institutionalized"":...",0,NaN


In [19]:
reviewed_df = gemini_df.copy()

In [20]:
TARGET_LABELS = [
    "ideological",
    "institutionalized",
    "interpersonal",
    "internalized"
]

train_df = reviewed_df.copy()

X = train_df["sentence"].astype(str)
Y = train_df[TARGET_LABELS].values
groups = train_df["source_id"].values

print("Training rows:", len(train_df))
print("Label counts:")
print(train_df[TARGET_LABELS].sum())

Training rows: 154
Label counts:
ideological          22
institutionalized    13
interpersonal        26
internalized         20
dtype: int64


In [21]:
splitter = GroupShuffleSplit(
    n_splits = 1,
    test_size = 0.25,
    random_state= 42
)

train_idx, test_idx = next(splitter.split(X,Y, groups = groups))

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]
Y_train = Y[train_idx]
Y_test = Y[test_idx]

print("Train:", len(X_train))
print("Test:", len(X_test))

Train: 107
Test: 47


In [22]:
baseline_model = Pipeline([
    ("tfidf", TfidfVectorizer(
        lowercase=True,
        ngram_range=(1, 2),
        min_df=1,
        max_df=0.95
    )),
    ("clf", OneVsRestClassifier(
        LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            solver="liblinear"
        )
    ))
])

In [23]:
baseline_model.fit(X_train, Y_train)
print("Model trained.")

Model trained.


In [25]:

Y_prob = baseline_model.predict_proba(X_test)

prob_df = pd.DataFrame(
    Y_prob,
    columns=[f"{label}_score" for label in TARGET_LABELS]
)

prob_df["sentence"] = X_test.values

prob_df.head(20)

print(classification_report(
    Y_test,
    Y_pred,
    target_names=TARGET_LABELS,
    zero_division=0
))

                   precision    recall  f1-score   support

      ideological       0.00      0.00      0.00         3
institutionalized       0.00      0.00      0.00         3
    interpersonal       0.00      0.00      0.00         7
     internalized       0.00      0.00      0.00         8

        micro avg       0.00      0.00      0.00        21
        macro avg       0.00      0.00      0.00        21
     weighted avg       0.00      0.00      0.00        21
      samples avg       0.00      0.00      0.00        21



: 